# Model Training Notebook

> **Note:** This notebook is for experimentation only.
> For the production pipeline, run `python -m src.train_model` from the project root.

FIX: Original notebook was incomplete (missing model.fit(), broken imports, wrong target column name).

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from utils.config import PROCESSED_DATA_PATH, TARGET_COLUMN, RANDOM_STATE

In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X = pd.get_dummies(X, drop_first=True)
X = X.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
print('Training complete')

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred, zero_division=0):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')